# 🎧 ทดสอบ TTS ไทย รอบ 2 — OmniVoice-Thai / F5-V2 / ThonburianTTS

รอบแรก edge-tts ชนะด้านความแม่น รอบนี้เอาตัวหนักที่ยังไม่ได้ลงสนามมาสู้ ใช้ประโยคชุดเดิม 6 ประโยคเทียบกันตรง ๆ

**วิธีใช้:** Runtime → Change runtime type → **T4 GPU** → Save แล้ว Runtime → **Run all** → รอ ~15 นาที → ฟังท้ายไฟล์

| เอนจิน | คืออะไร | ไลเซนส์ |
|---|---|---|
| omni_thai | OmniVoice ไฟน์จูนไทย (ฐาน Qwen3-0.6B, 600+ ภาษา) | Apache-2.0 ✅ ใช้เชิงพาณิชย์ได้ |
| omni_design | OmniVoice ออกแบบเสียงจากคำบรรยาย (ไม่ใช้ไฟล์อ้างอิง) | Apache-2.0 ✅ |
| f5thai_v2 | F5-TTS-THAI รุ่น V2 (IPA) — เคลมอ่านแม่นกว่า V1 ที่ฟังรอบแรก | โค้ด MIT / เช็ค weights |
| thonburian | ThonburianTTS (งานวิจัย biodatlab) | ⚠️ CC BY-NC-SA — **ฟังเทียบเท่านั้น ห้ามใช้ในคลิปมีรายได้** |

In [ ]:
# 1) ติดตั้ง (~3-4 นาที)
!pip install -q omnivoice f5-tts-th edge-tts soundfile
!pip install -q git+https://github.com/biodatlab/thonburian-tts.git || echo '⚠️ thonburian ติดตั้งไม่ได้ — เซลล์ของมันจะถูกข้ามเอง'
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'ไม่มี — แนะนำเปิด T4 ไม่งั้นช้ามาก')

In [ ]:
# 2) ประโยคทดสอบ (ชุดเดียวกับรอบแรก) + สร้างเสียงอ้างอิงสำหรับโคลน
import os, time
os.makedirs('samples2', exist_ok=True)

SENTENCES = {
  '01_basic': 'สวัสดีครับ ยินดีต้อนรับเข้าสู่ช่องของเรา วันนี้มีเรื่องราวน่าสนใจมาเล่าให้ฟังกันครับ',
  '02_story': 'ลองจินตนาการดูนะครับ ชายคนหนึ่งตัดสินใจเดินออกจากช่องยูทูบที่มีผู้ติดตามยี่สิบล้านคน โดยไม่บอกใครแม้แต่คำเดียว',
  '03_numbers_raw': 'เมื่อเวลา 18:30 น. ของวันที่ 2 สิงหาคม 2569 ยอดวิวพุ่งไปถึง 2,500,000 วิว สร้างรายได้กว่า 64,000 บาท',
  '04_english_mix': 'ช่อง YouTube ของเราใช้ AI ทำคลิป Shorts ลง TikTok และ Facebook ทุกวัน',
  '05_english_thai_spelled': 'ช่องยูทูบของเราใช้เอไอทำคลิปช็อตส์ ลงติ๊กต่อกและเฟซบุ๊กทุกวัน',
  '06_emotion': 'อะไรนะครับ! เขาทิ้งรายได้เดือนละหลายล้านบาทไปเฉย ๆ อย่างนั้นเหรอ เรื่องนี้มันบ้ามาก ๆ ไม่มีใครเชื่อเลยจริง ๆ',
}
timing = {}

# เสียงอ้างอิง: เปรมวดีพูดประโยคแรก (อยากใช้เสียงตัวเอง: อัปโหลด .wav ~10 วิ แล้วเปลี่ยน 2 ตัวแปรนี้)
import edge_tts
await edge_tts.Communicate(SENTENCES['01_basic'], 'th-TH-PremwadeeNeural').save('ref_premwadee.mp3')
!ffmpeg -y -loglevel error -i ref_premwadee.mp3 -ar 24000 -ac 1 ref_premwadee.wav
REF_AUDIO = 'ref_premwadee.wav'
REF_TEXT = SENTENCES['01_basic']
print('พร้อม: เสียงอ้างอิง + ', len(SENTENCES), 'ประโยค')

In [ ]:
# 3) OmniVoice-Thai — ไฟน์จูนไทยของ OmniVoice (ตัวเก็งด้านไลเซนส์: Apache-2.0)
assert 'SENTENCES' in globals(), '⚠️ ยังไม่ได้รันเซลล์ 2 — กด ▶ เซลล์ 2 ก่อน'
try:
    import soundfile as sf
    from omnivoice import OmniVoice

    dev = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    dt = torch.float16 if torch.cuda.is_available() else torch.float32
    MODEL_ID = 'hotdogs/omnivoice-thai'  # ถ้าโหลดไม่ได้ ถอยไปตัวฐาน k2-fsa/OmniVoice
    try:
        model = OmniVoice.from_pretrained(MODEL_ID, device_map=dev, dtype=dt)
    except Exception as e:
        print('โหลดตัวไฟน์จูนไทยไม่ได้ ->', e, '\nใช้ตัวฐานแทน')
        MODEL_ID = 'k2-fsa/OmniVoice'
        model = OmniVoice.from_pretrained(MODEL_ID, device_map=dev, dtype=dt)
    tag = 'omni_thai' if 'hotdogs' in MODEL_ID else 'omni_base'
    print('ใช้โมเดล:', MODEL_ID)

    for sid, text in SENTENCES.items():
        try:
            t0 = time.time()
            audio = model.generate(text=text, ref_audio=REF_AUDIO, ref_text=REF_TEXT)
            sf.write(f'samples2/{tag}_{sid}.wav', audio[0], 24000)
            timing[f'{tag}_{sid}'] = time.time() - t0
            print(f'เสร็จ: {sid} ({timing[f"{tag}_{sid}"]:.1f} วิ)')
        except Exception as e:
            print('ข้าม', sid, '->', e)

    # โบนัส: ออกแบบเสียงใหม่จากคำบรรยาย ไม่ต้องมีไฟล์อ้างอิง
    try:
        audio = model.generate(text=SENTENCES['02_story'], instruct='male, deep warm voice, storytelling narrator')
        sf.write('samples2/omni_design_02_story.wav', audio[0], 24000)
        print('เสร็จ: voice design (เสียงชายเล่าเรื่อง สร้างจากคำบรรยายล้วน ๆ)')
    except Exception as e:
        print('ข้าม voice design ->', e)

    del model
    torch.cuda.empty_cache()
except Exception as e:
    print('ข้าม OmniVoice เพราะ:', e)

In [ ]:
# 4) F5-TTS-THAI รุ่น V2 (IPA) — คู่เทียบตรงกับ V1 ที่ฟังไปรอบแรก
assert 'SENTENCES' in globals(), '⚠️ ยังไม่ได้รันเซลล์ 2 — กด ▶ เซลล์ 2 ก่อน'
try:
    import soundfile as sf
    from f5_tts_th.tts import TTS

    todo = list(SENTENCES.items())
    if not torch.cuda.is_available():
        todo = todo[:2]
        print('⚠️ ไม่มี GPU — เจนแค่ 2 ประโยคแรก')

    tts = TTS(model='v2')
    for sid, text in todo:
        t0 = time.time()
        wav = tts.infer(ref_audio=REF_AUDIO, ref_text=REF_TEXT, gen_text=text, step=32, cfg=2.0, speed=1.0)
        sf.write(f'samples2/f5thai_v2_{sid}.wav', wav, 24000)
        timing[f'f5thai_v2_{sid}'] = time.time() - t0
        print(f'เสร็จ: {sid} ({timing[f"f5thai_v2_{sid}"]:.1f} วิ)')
    del tts
    torch.cuda.empty_cache()
except Exception as e:
    print('ข้าม F5-V2 เพราะ:', e)

In [ ]:
# 5) ThonburianTTS (megaF5) — งานวิจัยไทย ⚠️ โมเดล CC BY-NC-SA: ฟังเทียบเท่านั้น ห้ามใช้เชิงพาณิชย์
assert 'SENTENCES' in globals(), '⚠️ ยังไม่ได้รันเซลล์ 2 — กด ▶ เซลล์ 2 ก่อน'
try:
    from flowtts.inference import FlowTTSPipeline, ModelConfig, AudioConfig

    model_config = ModelConfig(
        language='th', model_type='F5',
        checkpoint='hf://biodatlab/ThonburianTTS/megaF5/mega_f5_last.safetensors',
        vocab_file='hf://biodatlab/ThonburianTTS/megaF5/mega_vocab.txt',
        vocoder='vocos',
        device='cuda' if torch.cuda.is_available() else 'cpu',
    )
    audio_config = AudioConfig(silence_threshold=-45, cfg_strength=2.5, speed=1.0)
    pipeline = FlowTTSPipeline(model_config, audio_config)

    for sid, text in SENTENCES.items():
        try:
            t0 = time.time()
            pipeline(text=text, ref_voice=REF_AUDIO, ref_text=REF_TEXT,
                     output_file=f'samples2/thonburian_{sid}.wav')
            timing[f'thonburian_{sid}'] = time.time() - t0
            print(f'เสร็จ: {sid} ({timing[f"thonburian_{sid}"]:.1f} วิ)')
        except Exception as e:
            print('ข้าม', sid, '->', e)
    del pipeline
    torch.cuda.empty_cache()
except Exception as e:
    print('ข้าม ThonburianTTS เพราะ:', e)

In [ ]:
# 6) 🎧 ฟังเทียบรอบ 2 — รันเซลล์นี้ซ้ำทุกครั้งที่เจนเสียงเพิ่ม
import glob, os
from IPython.display import Audio, display, Markdown

for sid in SENTENCES:
    display(Markdown(f'---\n### {sid}\n> {SENTENCES[sid]}'))
    for f in sorted(glob.glob(f'samples2/*_{sid}.*')):
        if os.path.getsize(f) == 0:
            continue
        name = f.split('/')[-1]
        secs = timing.get(name.rsplit('.', 1)[0])
        display(Markdown(f'**{name}**' + (f' — {secs:.1f} วิ' if secs else '')))
        display(Audio(f))

In [ ]:
# 7) (ทางเลือก) รวมเป็น zip
!zip -q -r thai_tts_samples_round2.zip samples2
try:
    from google.colab import files
    files.download('thai_tts_samples_round2.zip')
except Exception as e:
    print('ดาวน์โหลดอัตโนมัติไม่ได้:', e)
    print('ดึงเองได้ที่แถบ Files (ไอคอนโฟลเดอร์ 📁 ซ้ายมือ)')